# Proyecto VPC: Adversarial Camouflage Aerial
Notebook principal para ejecutar el proyecto en Google Colab con GPU.
Incluye setup, validación del entorno, carga del modelo YOLOv8 OBB, preparación de datos, experimentos y evaluación.

## Estado Del Avance

Este notebook queda definido como el archivo principal de trabajo para ejecutar el proyecto desde Google Colab.

Avances actuales:

- El repositorio ya tiene setup de Colab, configuración CUDA y validación de runtime.
- Se agregó un primer conjunto de labels/anotaciones DOTA v1.0 al repositorio como avance inicial del dataset.
- Las imágenes DOTA ya fueron conseguidas, pero por tamaño no se versionan en Git. Se subirán a Google Drive.

Ruta oficial de Drive para datos y resultados:

`/content/drive/MyDrive/Proyecto-VPC/`

Estructura esperada para DOTA v1:

- Zips de imágenes: `/content/drive/MyDrive/Proyecto-VPC/data/raw/dota_v1/zips/`
- Imágenes extraídas: `/content/drive/MyDrive/Proyecto-VPC/data/raw/dota_v1/images/`
- Labels extraídos: `/content/drive/MyDrive/Proyecto-VPC/data/raw/dota_v1/labels/`

Siguiente objetivo: correr un baseline con YOLOv8 OBB sobre imágenes limpias antes de generar la textura/camuflaje adversarial.

## 1. Verificar GPU De Colab

Confirma que Colab asignó una GPU antes de instalar dependencias o correr YOLO.

In [ ]:
!nvidia-smi

## 2. Clonar O Actualizar El Repositorio

In [ ]:
%cd /content
!rm -rf Proyecto-VPC
!git clone https://github.com/Ctribsz/Proyecto-VPC.git
%cd Proyecto-VPC
!git log --oneline -2

/content
Cloning into 'Proyecto-VPC'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 78 (delta 14), reused 75 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 21.16 KiB | 4.23 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Proyecto-VPC
7567063 (HEAD -> main, origin/main, origin/HEAD) Add Colab setup
696eecd Init project


## 3. Instalar Dependencias

In [ ]:
!pip install -r requirements.txt

Obtaining file:///content/Proyecto-VPC
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adversarial-camouflage-aerial (pyproject.toml) ... done
  Created wheel for adversarial-camouflage-aerial: filename=adversarial_camouflage_aerial-0.1.0-0.editable-py3-none-any.whl size=3076 sha256=a94608563b30caea5615025105bb83b52c29d0fa2a6c55074bb254bb36d04d8b
  Stored in directory: /tmp/pip-ephem-wheel-cache-78i5jb__/wheels/a2/3d/1a/6f86d4264f63c0a49eaa9f509bb68357bfa3ac8cc4bbdcdc96
Successfully built adversarial-camouflage-aerial
  Attempting uninstall: adversarial-camouflage-aerial
    Found existing installation: adversarial-camouflage-aerial 0.1.0
    Uninstalling adversarial-camouflage-aerial-0.1.0:
      Successfully uninstalled adversarial-camouflage-aerial-0.1.0


## 4. Verificar Runtime Del Proyecto

In [ ]:
!python scripts/check_runtime.py

Python: 3.12.13
torch: 2.10.0+cu128
cuda available: True
cuda device: Tesla T4
cuda memory: 14.6 GiB
ultralytics: 8.4.45
kornia: 0.8.2
albumentations: 2.0.8
camouflage: not installed


## 5. Validar Configuración De Colab

In [ ]:
!python scripts/train_attack.py --config src/camouflage/config/colab.yaml
!python scripts/eval_digital.py --config src/camouflage/config/colab.yaml
!python scripts/eval_physical.py --config src/camouflage/config/colab.yaml

Loaded config for project: adversarial-camouflage-aerial-colab
Training skeleton is ready. Pass --run after implementing AttackTrainer.run().
Loaded config for digital evaluation: adversarial-camouflage-aerial-colab
Evaluation skeleton is ready. Pass --run after implementing EvaluationRunner.run().
Loaded config for physical evaluation: adversarial-camouflage-aerial-colab
Evaluation skeleton is ready. Pass --run after implementing EvaluationRunner.run().


## 6. Probar YOLOv8 OBB

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n-obb.pt")
print("Modelo cargado correctamente")
print(model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Modelo cargado correctamente
{0: 'plane', 1: 'ship', 2: 'storage tank', 3: 'baseball diamond', 4: 'tennis court', 5: 'basketball court', 6: 'ground track field', 7: 'harbor', 8: 'bridge', 9: 'large vehicle', 10: 'small vehicle', 11: 'helicopter', 12: 'roundabout', 13: 'soccer ball field', 14: 'swimming pool'}


## 7. Montar Google Drive

Drive se usa para leer las imágenes DOTA y guardar resultados generados sin versionarlos en Git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 8. Crear Estructura De Carpetas DOTA v1

Crea las carpetas oficiales en Drive para zips, imágenes extraídas, labels y resultados baseline.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Proyecto-VPC')
DOTA_ROOT = DRIVE_ROOT / 'data/raw/dota_v1'
ZIP_DIR = DOTA_ROOT / 'zips'
IMAGE_DIR = DOTA_ROOT / 'images'
LABEL_DIR = DOTA_ROOT / 'labels'
BASELINE_DIR = DRIVE_ROOT / 'results/digital/baseline_yolo_obb'

for path in [ZIP_DIR, IMAGE_DIR, LABEL_DIR, BASELINE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
    print(path)

## 9. Verificar Dataset En Drive

Este dry run no ejecuta YOLO. Solo revisa cuántas imágenes, labels y zips hay en las rutas esperadas.

Si `Images found` sale `0`, primero hay que subir y extraer las imágenes DOTA en Drive.

In [ ]:
!python scripts/run_yolo_baseline.py --config src/camouflage/config/colab.yaml --dry-run

## 10. Ejecutar Baseline YOLOv8 OBB

Corre YOLO sobre imágenes limpias para comprobar que el modelo detecta objetos antes de intentar el camuflaje adversarial.

Empezamos con 10 imágenes para que la prueba sea rápida y revisable.

In [ ]:
!python scripts/run_yolo_baseline.py --config src/camouflage/config/colab.yaml --max-images 10

## 11. Revisar Resultados Baseline

Muestra dónde quedaron las predicciones y abre una imagen anotada si existe.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

BASELINE_DIR = Path('/content/drive/MyDrive/Proyecto-VPC/results/digital/baseline_yolo_obb')
PREDICTIONS_PATH = BASELINE_DIR / 'predictions.json'
PLOTS_DIR = BASELINE_DIR / 'plots'
plots = sorted(PLOTS_DIR.glob('*')) if PLOTS_DIR.exists() else []

print(f'Predicciones: {PREDICTIONS_PATH}')
print(f'Visualizaciones encontradas: {len(plots)}')
for path in plots[:10]:
    print(path)

if plots:
    display(Image(filename=str(plots[0])))
else:
    print('No hay visualizaciones todavía; primero ejecuta el baseline.')